In [ ]:
!git clone https://github.com/dvhanh-code/netiquette-multilabel-classification.git
%cd netiquette-multilabel-classification

Cloning into 'netiquette-multilabel-classification'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 302 (delta 34), reused 66 (delta 17), pack-reused 204 (from 2)
Receiving objects: 100% (302/302), 219.59 MiB | 16.61 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (99/99), done.
/content/netiquette-multilabel-classification


In [ ]:
!pip install transformers datasets accelerate sentencepiece \
              scikit-learn iterative-stratification \
              pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"

In [ ]:
!mkdir -p data/final

!ln -s \
"/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet" \
"data/final/unified_final_v1.parquet"

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Mount Drive + Clone repo + Install
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/dvhanh-code/netiquette-multilabel-classification
%cd netiquette-multilabel-classification

!pip install transformers torch pandas numpy scikit-learn -q

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Enable GPU: Runtime → Change runtime type → L4")
print(f" GPU:  {torch.cuda.get_device_name(0)}")
print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'netiquette-multilabel-classification'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 302 (delta 34), reused 66 (delta 17), pack-reused 204 (from 2)
Receiving objects: 100% (302/302), 219.59 MiB | 18.52 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (99/99), done.
/content/netiquette-multilabel-classification/netiquette-multilabel-classification
 GPU:  NVIDIA L4
 VRAM: 23.7 GB


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Setup data
# ═══════════════════════════════════════════════════════════
import os

os.makedirs("data/final", exist_ok=True)

src = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"
dst = "data/final/unified_final_v1.parquet"

if not os.path.exists(dst):
    os.symlink(src, dst)

if not os.path.exists(dst):
    raise FileNotFoundError(f"Dataset not found: {dst}")

import pandas as pd
df = pd.read_parquet(dst)
print(f" Dataset: {len(df):,} rows")
print(df["split"].value_counts())

 Dataset: 453,242 rows
split
train    426742
test      13250
val       13250
Name: count, dtype: int64


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell: gbert large gold_silver 128 + MaskedFocalLoss
#       + Early Stopping + Per-label alpha
# ═══════════════════════════════════════════════════════════
import os, json, random, shutil
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from src.training.transformer_dataset import (
    LABELS, NetiquetteTransformerDataset,
    load_dataset, print_dataset_summary)
from src.training.losses import MaskedFocalLoss
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table)

# ── Config ─────────────────────────────────────────────────
MODEL_NAME   = "deepset/gbert-large"
MODE         = "gold_only"
MAX_LEN      = 128
BATCH        = 8
EPOCHS       = 5
LR           = 1e-5
WARMUP_RATIO = 0.06
PATIENCE     = 2
GAMMA        = 2.0
LOCAL_OUT    = "results/gbert_large_gold_only_128_focal"
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_only_128_focal_batch8"
DATA_PATH    = "data/final/unified_final_v1.parquet"

# ── Pre-flight ─────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("No GPU!")
device = torch.device("cuda")
print(f" GPU:  {torch.cuda.get_device_name(0)}")
print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ── Compute per-label alpha từ training data ───────────────
print("\nComputing per-label alpha...")
df_full = __import__("pandas").read_parquet(DATA_PATH)
train_df = df_full[df_full["split"] == "train"].copy()

alpha_values = []
for label in LABELS:
    ann = train_df[label].notna()
    pos = (train_df.loc[ann, label] == 1.0).sum()
    neg = (train_df.loc[ann, label] == 0.0).sum()
    # alpha_pos = neg/(pos+neg) → hiếm hơn → weight cao hơn
    alpha_pos = neg / (pos + neg) if (pos + neg) > 0 else 0.5
    alpha_values.append(float(alpha_pos))
    print(f"  {label:<12}: pos={pos:>7,}  neg={neg:>7,}  alpha={alpha_pos:.4f}")

alpha_tensor = torch.tensor(alpha_values, dtype=torch.float32)
print(f"\nAlpha vector: {alpha_values}")

# ── Save config ────────────────────────────────────────────
json.dump({
    "model_name": MODEL_NAME, "mode": MODE,
    "max_length": MAX_LEN,   "batch_size": BATCH,
    "epochs": EPOCHS,        "patience": PATIENCE,
    "learning_rate": LR,     "warmup_ratio": WARMUP_RATIO,
    "loss": "MaskedFocalLoss",
    "gamma": GAMMA,          "alpha": alpha_values,
    "optimizer": "AdamW",    "weight_decay": 0.01,
    "seed": 42,
}, open(f"{LOCAL_OUT}/config.json", "w"), indent=2, ensure_ascii=False)

print(f"\nModel:        {MODEL_NAME}")
print(f"Mode:         {MODE}")
print(f"Max length:   {MAX_LEN}")
print(f"Batch size:   {BATCH}")
print(f"Loss:         MaskedFocalLoss (gamma={GAMMA})")
print(f"Early stop:   patience={PATIENCE}")
print(f"Drive output: {DRIVE_OUT}")

# ── Sync function ──────────────────────────────────────────
def sync_to_drive():
    for fname in ["config.json", "summary_partial.json",
                  "val_metrics_latest.csv", "val_tuned_latest.csv",
                  "thresholds.json", "test_metrics.csv", "summary.json"]:
        src = f"{LOCAL_OUT}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")
    for fname in os.listdir(LOCAL_OUT):
        if (fname.startswith("val_metrics_epoch_") or
            fname.startswith("val_tuned_epoch_")) and fname.endswith(".csv"):
            shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")
    local_best = f"{LOCAL_OUT}/best_model"
    drive_best = f"{DRIVE_OUT}/best_model"
    if os.path.exists(local_best) and os.listdir(local_best):
        os.makedirs(drive_best, exist_ok=True)
        for fname in os.listdir(local_best):
            shutil.copy2(f"{local_best}/{fname}", f"{drive_best}/{fname}")
    print("   Synced to Drive")

# ── Model ──────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = out.pooler_output if (
            hasattr(out, "pooler_output") and out.pooler_output is not None
        ) else out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))

# ── Setup ──────────────────────────────────────────────────
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)

splits    = load_dataset(DATA_PATH, mode=MODE)
print_dataset_summary(splits)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

train_loader = make_loader(splits["train"], True)
val_loader   = make_loader(splits["val"],   False)
test_loader  = make_loader(splits["test"],  False)

model   = TransformerClassifier(MODEL_NAME).to(device)
loss_fn = MaskedFocalLoss(alpha=alpha_tensor, gamma=GAMMA)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps)

print(f"\nSteps/epoch: {len(train_loader):,}")
print(f"Total steps: {total_steps:,}")
print(f"Warmup:      {int(total_steps * WARMUP_RATIO):,}")

sync_to_drive()

# ── Training loop mit Early Stopping ──────────────────────
best_val_s    = -1.0
best_epoch    = -1
patience_left = PATIENCE
epoch_results = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*60}\nEPOCH {epoch}/{EPOCHS}  (patience left: {patience_left})\n{'='*60}")

    # Train
    model.train()
    total_loss, steps = 0.0, 0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(batch["input_ids"], batch["attention_mask"],
                       batch.get("token_type_ids"))
        loss = loss_fn(logits, batch["labels"], batch["label_mask"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item(); steps += 1
        if step % 500 == 0:
            print(f"  step {step}/{len(train_loader):,} loss={total_loss/steps:.4f}")

    train_loss = total_loss / steps
    print(f"\nEpoch {epoch} train loss: {train_loss:.4f}")

    # Validate
    model.eval()
    all_l, all_lb, all_m = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch.get("token_type_ids"))
            all_l.append(logits.cpu().numpy())
            all_lb.append(batch["labels"].cpu().numpy())
            all_m.append(batch["label_mask"].cpu().numpy())

    val_logits = np.concatenate(all_l)
    val_labels = np.concatenate(all_lb)
    val_masks  = np.concatenate(all_m)

    # threshold=0.5
    val_metrics = compute_multilabel_metrics(
        val_logits, val_labels, val_masks, split_name="val")
    print_metrics_table(f"VAL Epoch {epoch} threshold=0.5", val_metrics)

    # tuned threshold
    tuned = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
    print(f"\n  Tuned thresholds: {tuned}")
    val_tuned = compute_multilabel_metrics(
        val_logits, val_labels, val_masks,
        thresholds=tuned, split_name="val_tuned")
    print_metrics_table(f"VAL Epoch {epoch} TUNED", val_tuned)

    macro_s = float(
        val_tuned[val_tuned["label"] == "MACRO"]["s_score"].iloc[0])

    # Save metrics
    val_metrics.to_csv(f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv", index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_epoch_{epoch}.csv",    index=False)
    val_metrics.to_csv(f"{LOCAL_OUT}/val_metrics_latest.csv",        index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_latest.csv",            index=False)

    epoch_results.append({
        "epoch": epoch, "train_loss": train_loss, "val_macro_s": macro_s})

    # Early stopping logic
    if macro_s > best_val_s:
        best_val_s    = macro_s
        best_epoch    = epoch
        patience_left = PATIENCE   # reset patience
        torch.save(model.state_dict(),
                   f"{LOCAL_OUT}/best_model/pytorch_model.bin")
        tokenizer.save_pretrained(f"{LOCAL_OUT}/best_model")
        print(f"\n  → New best: S={macro_s:.4f} | patience reset to {PATIENCE}")
    else:
        patience_left -= 1
        print(f"\n  → No improvement. patience left: {patience_left}")
        if patience_left <= 0:
            print(f"\n  ⏹ Early stopping triggered at epoch {epoch}!")
            json.dump({
                "model_name": MODEL_NAME, "mode": MODE,
                "epoch": epoch, "best_epoch": best_epoch,
                "best_val_s": best_val_s, "epochs_done": epoch_results,
                "status": "early_stopped",
            }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
               indent=2, ensure_ascii=False)
            sync_to_drive()
            break

    json.dump({
        "model_name": MODEL_NAME, "mode": MODE,
        "epoch": epoch, "best_epoch": best_epoch,
        "best_val_s": best_val_s, "epochs_done": epoch_results,
        "status": "training",
    }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
       indent=2, ensure_ascii=False)

    sync_to_drive()

# ── Final evaluation ───────────────────────────────────────
print(f"\nBest epoch: {best_epoch}  |  val S={best_val_s:.4f}")

model.load_state_dict(torch.load(
    f"{LOCAL_OUT}/best_model/pytorch_model.bin",
    map_location=device, weights_only=True))
model.eval()

def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())
    return np.concatenate(ll), np.concatenate(lb), np.concatenate(lm)

val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(
    val_logits, val_labels, val_masks, metric="s_score")
print("Final thresholds:", best_thresholds)

test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test")
print_metrics_table("FINAL TEST METRICS", test_metrics)

test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)
json.dump(best_thresholds,
          open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)
json.dump({
    "model_name": MODEL_NAME, "mode": MODE,
    "max_length": MAX_LEN,   "batch_size": BATCH,
    "loss": "MaskedFocalLoss", "gamma": GAMMA, "alpha": alpha_values,
    "best_epoch": best_epoch,"best_val_s": best_val_s,
    "epochs_done": epoch_results, "status": "done",
    "test_macro": test_metrics[
        test_metrics["label"] == "MACRO"].iloc[0].to_dict(),
}, open(f"{LOCAL_OUT}/summary.json", "w"),
   indent=2, ensure_ascii=False)

sync_to_drive()
print(f"\n Done! Results at:\n  {DRIVE_OUT}")

 GPU:  NVIDIA L4
 VRAM: 23.7 GB

Computing per-label alpha...
  hate_speech : pos=  8,637  neg=269,878  alpha=0.9690
  toxic       : pos= 34,936  neg=305,831  alpha=0.8975
  threat      : pos=    735  neg=235,964  alpha=0.9969
  insult      : pos= 19,372  neg=266,942  alpha=0.9323

Alpha vector: [0.9689891029208481, 0.8974783356369601, 0.9968947904300398, 0.932340018301585]

Model:        deepset/gbert-large
Mode:         gold_only
Max length:   128
Batch size:   8
Loss:         MaskedFocalLoss (gamma=2.0)
Early stop:   patience=2
Drive output: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_batch8

Dataset summary:

TRAIN:
  rows   : 61,830
  gold   : 61,830
  silver : 0
  hate_speech  annotated= 61,830 pos= 6,635 neg= 55,195
  toxic        annotated= 12,735 pos= 3,819 neg=  8,916
  threat       annotated= 20,014 pos=    96 neg= 19,918
  insult       annotated= 32,749 pos= 6,050 neg= 26,699

VAL:
  rows   : 13,250
  gold   : 13,250
  silver : 0
  hate_spe

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Steps/epoch: 7,729
Total steps: 38,645
Warmup:      2,318
   Synced to Drive

EPOCH 1/5  (patience left: 2)
  step 500/7,729 loss=0.0240
  step 1000/7,729 loss=0.0216
  step 1500/7,729 loss=0.0203
  step 2000/7,729 loss=0.0198
  step 2500/7,729 loss=0.0192
  step 3000/7,729 loss=0.0188
  step 3500/7,729 loss=0.0187
  step 4000/7,729 loss=0.0186
  step 4500/7,729 loss=0.0185
  step 5000/7,729 loss=0.0184
  step 5500/7,729 loss=0.0181
  step 6000/7,729 loss=0.0180
  step 6500/7,729 loss=0.0179
  step 7000/7,729 loss=0.0178
  step 7500/7,729 loss=0.0177

Epoch 1 train loss: 0.0176

VAL Epoch 1 threshold=0.5
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2     mcc  s_score  support_pos  support_total
hate_speech     0.5000     0.1146  0.9880 0.2054 0.3915  0.0830   0.4665         1422          13250
      toxic     0.5000     0.3689  0.9841 0.5366 0.7380  0.2991   0.6937          81